# Tune LR, LightGBM, XGBoost, NN, MARS models w/ optuna

## Set Up

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.data_utils import export_data
import pandas as pd
from src.tune import extract_metrics_from_log

Set globals

In [ ]:
MODEL_ABRV_LIST = [
    "lr",
    "lgbm",
    "xgb",
    "nn",
]
OUTCOME_LIST = [
    "SERIOUS",
    "ANY",
    "PNEUMO",
    "CARDIAC_COMP",
    "VTE",
    "SEPSIS",
    "SSI",
    "UTI",
    "RENAL",
    "UNPLNREOP",
    "MORT",
]
SCORING = "average_precision"
DATA_IMP_DIR = BASE_PATH / "data" / "processed"
N_CV_SPLITS = 3
N_CV_REPEATS = 10
SWARM_DIR = BASE_PATH / "swarm" / "tune"
CMD_DIR = SWARM_DIR / "commands"
LOG_DIR = SWARM_DIR / "logs"

In [ ]:
model_config_dict = {
    "lr": {
        "gb": 5,
        "swarm_time": "20:00:00",
        "n_trials": 150,
    },
    "xgb": {
        "gb": 20,
        "swarm_time": "7:00:00",
        "n_trials": 300,
    },
    "nn": {
        "gb": 30,
        "swarm_time": "1-8:00:00",
        "n_trials": 150,
    },
    "lgbm": {
        "gb": 20,
        "swarm_time": "7:00:00",
        "n_trials": 300,
    },
}

## RUN

In [ ]:
def make_cmd(
    model_abrv,
    outcome_name,
    scoring_str,
    import_dir,
    model_save_dir,
    n_repeats,
    n_trials,
    n_cv_splits,
    n_parallel_cv,
    seed,
    eval_bootstraps,
):
    cmd_str = f"export PYTHONPATH={BASE_PATH}; \
                export OMP_NUM_THREADS={int(n_parallel_cv*1.5)}; \
                uv run python -m src.tune \
                --model_abrv {model_abrv} \
                --outcome_name {outcome_name} \
                --scoring {scoring_str} \
                --import_dir {import_dir} \
                --n_trials {n_trials} \
                --n_cv_splits {n_cv_splits} \
                --n_repeats {n_repeats} \
                --n_parallel_cv {n_parallel_cv} \
                --eval_bootstraps {eval_bootstraps} \
                --seed {seed}"
    if model_save_dir is not None:
        cmd_str += f" --model_save_dir {model_save_dir} "
    return " ".join(cmd_str.split())


full_cmd_list = []
for model in MODEL_ABRV_LIST:
    swarm_path = CMD_DIR / f"{model}.swarm"
    if swarm_path.exists():
        swarm_path.unlink()
    swarm_path.parent.mkdir(exist_ok=True, parents=True)
    outcome_swarm_list = []
    for outcome in OUTCOME_LIST:
        model_save_dir = BASE_PATH / "models"
        cmd_str = make_cmd(
            model_abrv=model,
            outcome_name=outcome,
            scoring_str="average_precision",
            import_dir=DATA_IMP_DIR,
            model_save_dir=model_save_dir,
            n_trials=model_config_dict[model]["n_trials"],
            n_cv_splits=N_CV_SPLITS,
            n_repeats=N_CV_REPEATS,
            n_parallel_cv=N_CV_SPLITS * N_CV_REPEATS,
            seed=SEED,
            eval_bootstraps=5,
        )
        outcome_swarm_list.append(cmd_str)
    swarm_path.write_text("\n".join(outcome_swarm_list))
    full_cmd_list += outcome_swarm_list
print(f"Total commands: {len(full_cmd_list)}")

Locally sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     subprocess.run(cmd, shell=True, check=True)

On swarm

In [ ]:
# run swarm
from src.swarm import run_tune_swarm

run_tune_swarm(
    model_list=MODEL_ABRV_LIST,
    log_base_dir=LOG_DIR,
    cmd_dir=CMD_DIR,
    n_threads=N_CV_SPLITS * N_CV_REPEATS,
    config_dict=model_config_dict,
    partition="norm",
)

## Look at results

In [ ]:
log_dir = BASE_PATH / "swarm" / "tune" / "logs"
all_rows = []
for model_dir in log_dir.iterdir():
    model_name = model_dir.name
    print(f"{model_name}...")
    for log_path in model_dir.iterdir():
        if log_path.suffix != ".o":  # logs written to .e files
            continue
        cur_row = extract_metrics_from_log(log_path=log_path, model_name=model_name)
        all_rows.append(cur_row)
metric_df = pd.DataFrame(all_rows)
metric_df = metric_df.sort_values(
    by=["outcome", "bin_High"], ascending=[False, False]
).reset_index(drop=True)
export_data(
    data_to_export=metric_df, export_path=BASE_PATH / "results/tune_metrics.xlsx"
)
metric_df